# 9 · Distributed E91 / BBM92 (entanglement-based QKD) over FABRIC

Run **entanglement-based** QKD across two FABRIC nodes. Unlike BB84 (notebooks 2 and 7),
there is **no photon plane and no P4 switch**: Alice hosts a shared **quantum-state
service** (the entangled Bell-pair register), Bob measures his halves through a remote
proxy, and only the **classical coordination** — Alice's basis announcement, Bob's QBER
sample, and the CHSH Bell-test bits — rides the real data-plane link. That WAN path is the
same research lever as BB84; here it carries the entanglement protocol's coordination.

This is the `qne-sequence/` runtime (see `DESIGN.md` §6.3). Physics: a **Werner-state**
noise model with `w = fidelity` ties the key-error rate and the Bell violation together —
**QBER = (1−F)/2** and **CHSH S = 2√2·F** — so `S > 2` certifies entanglement and `S ≤ 2`
(at F ≤ 1/√2) means no secure key.

**Modes:** `e91` (adds the CHSH Bell test) or `bbm92` (Z/X only, key-efficient, security
from QBER like BB84). Fiber loss (distance/attenuation) is applied as **pair loss** in the
service, so those knobs still shape the detected-pair count.

### At a glance
- **Purpose:** run one distributed E91/BBM92 session across the slice and record the result.
- **Prereqs:** **run notebook 1 first** (slice + data-plane IPs). No BMv2/P4 switch needed here.
- **Inputs:** `SLICE_NAME`, `SCENARIO` (for F/distance/attenuation), and the knobs below.
- **Outputs:** `results/fabric_e91_{alice,bob}.json`; on-node logs at `/tmp/e91_{alice,bob}.log`.
- **Idempotent:** re-runnable; reuses the `.venv-qne` (sequence 1.0.0) built by notebook 7 / `setup_sequence_runtime`.

## 1 · Configuration

In [ ]:
SLICE_NAME = 'qfabric-bb84-2'                          # same slice as notebook 1
SCENARIO   = 'validation/scenarios/fabric_1km.yml'     # source of F / distance / attenuation

# --- Protocol ---
PROTOCOL   = 'e91'    # 'e91' (adds CHSH Bell test) | 'bbm92' (Z/X, key-efficient)

# Entanglement knobs
NUM_PAIRS       = 20000     # Bell pairs Alice generates
SAMPLE_FRACTION = 0.2       # fraction of sifted bits disclosed for QBER estimation
# Physics comes from the scenario (fidelity, distance, attenuation); override here if desired.

## 2 · Load the slice

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as df
from qne.config import ScenarioConfig

cfg = ScenarioConfig.from_yaml(PROJECT_DIR / SCENARIO)
fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

## 3 · Ship the latest code + build the SeQUeNCe runtime

`upload_project` re-tars the repo to every node (includes `qne-sequence/` with the new
entanglement modules). `setup_sequence_runtime` builds/reuses the `.venv-qne`
(Python 3.12 + `sequence==1.0.0`) on both nodes. Idempotent; skip if notebook 7 already ran it.

In [ ]:
df.upload_project(slice_obj)
df.setup_sequence_runtime(slice_obj)

## 4 · Run distributed E91 / BBM92

Bob listens (classical TCP); Alice connects over the data-plane IP, generates the Bell
pairs in her state service, and answers Bob's measurement RPCs. No switch, no root, no raw
sockets — entanglement bookkeeping is authoritative in the one service; only classical
coordination crosses the wire.

In [ ]:
a_res, b_res = df.run_sequence_e91(
    slice_obj,
    mode=PROTOCOL,
    num_pairs=NUM_PAIRS,
    fidelity=cfg.channel.polarization_fidelity,
    distance_km=cfg.channel.distance_km,
    attenuation=cfg.channel.attenuation_db_per_km,
    sample_fraction=SAMPLE_FRACTION,
)

## 5 · Verify

In [ ]:
import math

analytical_qber = (1.0 - cfg.channel.polarization_fidelity) / 2.0
analytical_chsh = 2 * math.sqrt(2) * cfg.channel.polarization_fidelity
print(f'analytical: QBER (1-F)/2 = {analytical_qber:.4f}, CHSH 2\u221a2\u00b7F = {analytical_chsh:.3f}\n')

checks = []
def rec(name, ok, detail=''):
    checks.append(ok)
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f' \u2014 {detail}' if detail else ''))

rec('alice produced a key', bool(a_res) and a_res.get('key') is not None)
rec('bob produced a key',   bool(b_res) and b_res.get('key') is not None)
if a_res:
    q = a_res.get('qber')
    rec('QBER plausible (< 0.11)', q is not None and 0 <= q < 0.11, f'QBER={q}')
    rec('secure fraction > 0', (a_res.get('secure_fraction') or 0) > 0,
        f"secure_fraction={a_res.get('secure_fraction')}")
    rec('loss removed some pairs', (a_res.get('detected_pairs') or 0) < NUM_PAIRS,
        f"detected={a_res.get('detected_pairs')}/{NUM_PAIRS}")
    if PROTOCOL == 'e91':
        s = a_res.get('chsh_s')
        rec('CHSH violates classical bound (S > 2 -> entanglement certified)',
            s is not None and s > 2.0, f'S={s}')

# cross-node consistency
if a_res and b_res:
    sa, sb = a_res.get('sifted_bits'), b_res.get('sifted_bits')
    rec('alice/bob agree on sifted count', sa is not None and sa == sb, f'alice={sa} bob={sb}')
    ka, kb = a_res.get('key_bits'), b_res.get('key_bits')
    rec('alice/bob agree on key length', ka is not None and ka == kb, f'alice={ka} bob={kb}')
    # no Cascade yet: keys differ only on error positions -> mismatch tracks QBER
    if a_res.get('key') is not None and b_res.get('key') is not None:
        q = a_res.get('qber') or 0.0
        nb = a_res.get('key_bits') or 1
        mism = bin(int(a_res['key']) ^ int(b_res['key'])).count('1') / nb
        if q == 0:
            rec('keys identical (QBER = 0)', mism == 0.0, f'mismatch={mism:.4f}')
        else:
            bound = q + 4 * math.sqrt(q * (1 - q) / nb) + 1 / nb
            rec('key mismatch tracks QBER (no Cascade yet)', mism <= bound,
                f'mismatch={mism:.4f} vs QBER={q:.4f} (bound {bound:.4f})')

print('\nALL CHECKS PASSED' if checks and all(checks)
      else '\nSOME CHECKS FAILED \u2014 inspect /tmp/e91_{alice,bob}.log on the nodes.')

## 6 · Notes & cleanup

- **No switch / no root:** entanglement has no photon plane — only the classical link is
  used, so this notebook needs neither BMv2 nor raw sockets. The `apply_classical_netem`
  lever still works: `df.apply_classical_netem(slice_obj, delay_ms=..., loss_pct=...)`
  impairs TCP:5100 to study how WAN conditions affect the Bell-test coordination. Clear
  with `df.clear_classical_netem(slice_obj)`.
- **Modes:** set `PROTOCOL='bbm92'` for a key-efficient Z/X run (no Bell test, ~2\u00d7 more
  sifted bits per pair); `PROTOCOL='e91'` adds the CHSH certification (fewer key bits, since
  the extreme angles are spent on the Bell test).
- **Keys track QBER, not bit-for-bit:** without error correction (Cascade, a planned
  extension) Alice's and Bob's keys differ on the error positions \u2014 the verify cell checks
  the mismatch tracks the QBER. A desynced run would show ~50%.
- **Sweep:** vary `SCENARIO` (distance/fidelity) and re-run cells 4\u20135; or run
  `qne-sequence/sweep.py` style matrices against `run_session` for slice-free curves.
- **Delete the slice** when done: `df.cleanup(fablib, SLICE_NAME)`.